# Project.py

In [3]:
import logging
logging.basicConfig(level=logging.INFO)

import warnings
warnings.filterwarnings(action='ignore')

In [4]:
from pathlib import Path
from src.project import PolymerBuildProject


project_path = Path('polyID_test')
# project_path = Path('polyid_scale')
project = PolymerBuildProject.get_project(project_path)

In [ ]:
project.print_status(
    detailed=True,
)

#### Run Jobs

In [ ]:
project.run()

In [ ]:
project.run(
    names=[
        # 'polymerize',
        # 'oligomerize',
        # 'pack_lattice',
        # 'to_interchange',
        'md_export'
    ],
    # jobs=[
    #     project.open_job(id='214e8512b6218c21668ce8c19c6bf359')
    # ]
)

### Inspect jobs

In [20]:
from polymerist.genutils.decorators.functional import allow_string_paths

@allow_string_paths
def ls(path : Path) -> None:
    if not path.is_dir():
        raise NotADirectoryError
    for subpath in path.iterdir():
        print(subpath.name)

@allow_string_paths
def cat(path : Path) -> None:
    if not path.is_file():
        raise IsADirectoryError
    
    with path.open('r') as file:
        for line in file.readlines():
            print(line, end='')

In [18]:
notable_jobs = [
    '214e8512b6218c21668ce8c19c6bf359', # stereochemistry is inverted by file write
    'a4b0a6be41f5786ff9f77f777cda59fa', # long alkane backbone hangs up partitions algorithm
    '41bc5b7d9700c043b78c0b898de4e758', # hangs with NAGL charges
]

In [ ]:
job = project.open_job(id=notable_jobs[-1])
project.get_job_status(job)

In [ ]:
offmol = load_job_oligomer_molecule(job)
offmol

In [ ]:
assign_partial_charges(job)

In [ ]:
cat(job.fn(PolymerBuildProject.LOGFILE_NAME))

In [24]:
cat(job.fn(PolymerBuildProject.OLIGOMER_SDF))

In [ ]:
ls(job.path)

In [ ]:
with open(job.fn(PolymerBuildProject.LOGFILE_NAME), 'r') as file:
    for line in file.readlines():
        print(line)

## Shipping completed jobs to NREL

In [8]:
ship_path = Path(f'{project_path.stem}_completed')
ship_path.mkdir(exist_ok=True)

projship = PolymerBuildProject.init_project(ship_path)

In [18]:
from shutil import copytree


for job in project:
    if exported_to_LAMMPS(job):
        copytree(job.path, f'{projship.workspace}/{job.id}')

# MD Engine file writing

## Eventual encapsulation class

In [ ]:
from abc import ABC, abstractmethod
from openff.interchange import Interchange
from polymerist.genutils.decorators.classmod import register_subclasses


@register_subclasses(key_attr='ENGINE')
class MDEngineExporter(ABC):
    '''For simplifying the process of '''
    def __init_subclass__(cls, **kwargs) -> None:
        '''Enforce class-level definition of "Engine" name attr in subclasses'''
        super().__init_subclass__()
        if not hasattr(cls, 'ENGINE'):
            raise NotImplementedError('No class attr "ENGINE" set for subclass')
        
    def __init__(self, interchange : Interchange) -> None:
        super().__init__()
        self.interchange = interchange

    @property
    def inc(self) -> Interchange:
        '''Alias of "self.interchange" for convenience'''
        return self.interchange
    
    @abstractmethod
    def write_inputs(*args, **kwargs) -> list[Path]:
        pass

    
# Concrete classes
class LAMMPSMDExporter(MDEngineExporter):
    ENGINE = 'LAMMPS'

class OpenMMMDExporter(MDEngineExporter):
    ENGINE = 'OpenMM'